# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [16]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)

# 0 for local model, 1 for deepseek-chat
env_to_load = 0

this_ai_model = None
this_base_url = None
this_api_key = None

local_ai_model = 'openai/gpt-oss-20b'
local_base_url = os.getenv('LMSTUDIO_BASE_URL')
local_api_key = os.getenv('LMSTUDIO_API_KEY')

deepseek_ai_model = 'deepseek-chat'
deepseek_base_url = os.getenv('DEEPSEEK_BASE_URL')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')


Add LMStudio as provider

In [17]:
match env_to_load:
    case 0:
        this_ai_model = local_ai_model
        this_base_url = local_base_url
        this_api_key = local_api_key
    case 1:
        this_ai_model = deepseek_ai_model
        this_base_url = deepseek_base_url
        this_api_key = deepseek_api_key
    case _:
        this_ai_model = local_ai_model
        this_base_url = local_base_url
        this_api_key = local_api_key 

llm = OpenAI(base_url=this_base_url, api_key=this_api_key)

In [18]:
# Set the system and user prompt
system_prompt = """
    You are a helpfull assistant specialized in python coding, helping a begginer programmer understand code.
    Your answers will be detailed, explaining the technical terms if used.
"""

user_prompt = """
    Please explain what this code does and why:
    yield from {book.get("author") for book in books if book.get("author")}
"""

In [19]:
# Get gpt-4o-mini to answer, with streaming
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [21]:
# Get Llama 3.2 to answer
llm_response = llm.chat.completions.create(model=local_ai_model, messages=messages, temperature=0.5)
#print(llm_response.choices[0].message.content)
display(Markdown(llm_response.choices[0].message.content))

Sure! Let’s walk through the line step by step and see what it actually does in a Python program.

```python
yield from {book.get("author") for book in books if book.get("author")}
```

---

## 1. The building blocks

| Symbol / keyword | What it is | How it works here |
|------------------|------------|-------------------|
| `books` | A **sequence** (list, tuple, etc.) of *book* objects – each one is a dictionary. | We loop over every element in this sequence. |
| `for book in books` | The classic Python **loop syntax** used inside comprehensions. | It pulls out one dictionary at a time and calls it `book`. |
| `book.get("author")` | A dictionary method that returns the value for key `"author"` or `None` if the key isn’t present. | We’re trying to read the author’s name from each book. |
| `{ … }` | Curly braces used for a **set comprehension** (just like `[ … ]` would be a list comprehension). | The expression inside creates a *new set* containing all unique authors we find. |
| `if book.get("author")` | A conditional filter that keeps only truthy values. | If the author field is missing, or it’s an empty string (`""`) or another “false” value, that book is skipped. |
| `yield from …` | A generator statement that **delegates** yielding to another iterable. | It takes whatever iterator we give it and yields each element one by one as if we had written a loop ourselves. |

---

## 2. What the set comprehension does

```python
{book.get("author") for book in books if book.get("author")}
```

1. **Iterate** over every `book` dictionary in `books`.
2. **Retrieve** the value of `"author"` with `.get()`.  
   - If the key is missing, `None` is returned.
3. **Filter** out any “falsy” values (`None`, empty string, 0, etc.) because of the `if book.get("author")`.
4. **Collect** the remaining author names into a **set**.

A set automatically removes duplicates, so if two books share the same author, that author will appear only once in the resulting set.

*Example:*  
If `books` is

```python
[
    {"title": "Book A", "author": "Alice"},
    {"title": "Book B", "author": "Bob"},
    {"title": "Book C"},                # no author key
    {"title": "Book D", "author": ""},  # empty string – treated as falsy
    {"title": "Book E", "author": "Alice"}   # duplicate author
]
```

the set comprehension yields:

```python
{"Alice", "Bob"}
```

---

## 3. What `yield from` does with that set

The expression inside `yield from` is an **iterable** (the set we just created).  
When a generator function contains

```python
yield from some_iterable
```

Python internally loops over `some_iterable` and yields each element to the caller of the generator. It’s equivalent to:

```python
for item in some_iterable:
    yield item
```

but written more concisely.

So, putting it all together, the single line

```python
yield from {book.get("author") for book in books if book.get("author")}
```

does these steps:

1. Builds a set of **unique author names** found in `books`, ignoring any missing or empty authors.
2. Yields each author name one by one to whatever consumes the generator.

---

## 4. Why use this pattern?

* **Conciseness:** One line replaces several lines of explicit loops and conditionals.
* **Readability for experienced readers:** Once you know the syntax, it’s clear that we’re “yielding all authors”.
* **Eliminates duplicates automatically** thanks to the set.
* **Graceful handling of missing data:** `.get()` avoids `KeyError`, and the `if` filter removes empty values.

---

## 5. A full example in context

```python
def unique_authors(books):
    """Yield each unique author from a list of book dicts."""
    yield from {book.get("author") for book in books if book.get("author")}
```

Usage:

```python
books = [
    {"title": "Book A", "author": "Alice"},
    {"title": "Book B", "author": "Bob"},
    {"title": "Book C"},                # no author key
    {"title": "Book D", "author": ""},  # empty string – ignored
    {"title": "Book E", "author": "Alice"}   # duplicate author
]

for author in unique_authors(books):
    print(author)
```

Output:

```
Alice
Bob
```

The generator yields `"Alice"` and `"Bob"` exactly once each, even though Alice appears twice in the input list.

---

### TL;DR

- **Set comprehension** collects all distinct, non‑empty author names from `books`.
- **`yield from`** streams those names out of a generator function one by one.
- The line is a compact way to “output every unique author” while gracefully handling missing or empty data.